# LESSON 4.7: Practical Applications and the FFT
## Filtering in the Frequency Domain

In this lesson:
- The Fast Fourier Transform (FFT) and its computational advantage
- Separability of the 2-D DFT
- Unsharp masking in the frequency domain
- Homomorphic filtering for illumination correction
- Complete image enhancement pipeline

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time

## 1. The Fast Fourier Transform (FFT)

### The Problem with the DFT

Recall the 1-D DFT definition:

$$F(u) = \sum_{x=0}^{M-1} f(x) \, e^{-j2\pi u x / M} \quad \text{for } u = 0, 1, \ldots, M-1$$

For each frequency $u$, we must sum over all $M$ spatial samples. Since there are $M$ frequencies, the total number of complex multiplications and additions is:

$$\text{DFT complexity} = O(M^2)$$

For a signal of length $M = 1024$, this means roughly $1,048,576$ operations.

### The FFT Solution

The **Fast Fourier Transform (FFT)**, discovered by Cooley and Tukey (1965), reduces this to:

$$\text{FFT complexity} = O(M \log_2 M)$$

For $M = 1024$: only $\approx 10,240$ operations -- a **100x speedup**!

### How does the FFT work?

The FFT exploits the **symmetry** and **periodicity** of the complex exponential $W_M = e^{-j2\pi/M}$:

1. **Symmetry**: $W_M^{k+M/2} = -W_M^k$
2. **Periodicity**: $W_M^{k+M} = W_M^k$

These properties allow the DFT to be recursively split into smaller DFTs (divide and conquer). A DFT of size $M$ is decomposed into two DFTs of size $M/2$, each of which is further split, until we reach DFTs of size 2.

### Speedup factor:

$$\text{Speedup} = \frac{M^2}{M \log_2 M} = \frac{M}{\log_2 M}$$

| $M$ | DFT ($M^2$) | FFT ($M\log_2 M$) | Speedup |
|-----|-------------|--------------------|---------|
| 32 | 1,024 | 160 | 6.4x |
| 256 | 65,536 | 2,048 | 32x |
| 1024 | 1,048,576 | 10,240 | 102x |
| 4096 | 16,777,216 | 49,152 | 341x |

In [ ]:
# Visualize the computational advantage of FFT over DFT
M_values = 2**np.arange(2, 14)  # 4, 8, 16, ..., 8192

dft_ops = M_values**2
fft_ops = M_values * np.log2(M_values)
speedup = dft_ops / fft_ops

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].semilogy(np.log2(M_values), dft_ops, 'ro-', linewidth=2, label='DFT: $O(M^2)$')
axes[0].semilogy(np.log2(M_values), fft_ops, 'bs-', linewidth=2, label='FFT: $O(M \log_2 M)$')
axes[0].set_xlabel('$\log_2(M)$', fontsize=12)
axes[0].set_ylabel('Number of Operations', fontsize=12)
axes[0].set_title('DFT vs FFT: Operation Count', fontsize=13)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

axes[1].plot(np.log2(M_values), speedup, 'g^-', linewidth=2, markersize=8)
axes[1].set_xlabel('$\log_2(M)$', fontsize=12)
axes[1].set_ylabel('Speedup Factor ($M / \log_2 M$)', fontsize=12)
axes[1].set_title('FFT Speedup Over DFT', fontsize=13)
axes[1].grid(True, alpha=0.3)

plt.suptitle('The Fast Fourier Transform: Computational Advantage', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Speedup examples:')
for m, s in zip(M_values, speedup):
    print(f'  M = {m:>5d}:  DFT = {m**2:>12,d} ops,  FFT = {int(m*np.log2(m)):>8,d} ops,  Speedup = {s:.1f}x')

### Timing Comparison: Manual DFT vs NumPy FFT

Let us verify the speedup experimentally. We implement the DFT directly from the formula and compare its execution time against NumPy's optimized FFT.

In [ ]:
def manual_dft_1d(f):
    """
    Compute the 1-D DFT directly from the definition.
    F(u) = sum_{x=0}^{M-1} f(x) * exp(-j*2*pi*u*x / M)
    
    Complexity: O(M^2)
    """
    M = len(f)
    F = np.zeros(M, dtype=np.complex128)
    for u in range(M):
        for x in range(M):
            F[u] += f[x] * np.exp(-1j * 2 * np.pi * u * x / M)
    return F

# Test correctness first on a small signal
test_signal = np.array([1.0, 2.0, 4.0, 3.0, 5.0, 2.0, 1.0, 3.0])
F_manual = manual_dft_1d(test_signal)
F_numpy = np.fft.fft(test_signal)

print('Correctness check (small signal, M=8):')
print(f'  Max difference between manual DFT and np.fft.fft: {np.max(np.abs(F_manual - F_numpy)):.2e}')
print('  Results match!\n')

In [ ]:
# Timing comparison for different signal lengths
sizes = [32, 64, 128, 256, 512]
times_manual = []
times_fft = []

print('Timing comparison (DFT vs FFT):')
print(f'{"M":>6s}  {"Manual DFT (s)":>15s}  {"NumPy FFT (s)":>15s}  {"Speedup":>10s}')
print('-' * 55)

for M in sizes:
    signal = np.random.randn(M)
    
    # Time manual DFT
    start = time.time()
    _ = manual_dft_1d(signal)
    t_manual = time.time() - start
    times_manual.append(t_manual)
    
    # Time NumPy FFT (average over many runs for accuracy)
    n_runs = 1000
    start = time.time()
    for _ in range(n_runs):
        np.fft.fft(signal)
    t_fft = (time.time() - start) / n_runs
    times_fft.append(t_fft)
    
    print(f'{M:>6d}  {t_manual:>15.6f}  {t_fft:>15.6f}  {t_manual/t_fft:>10.1f}x')

# Plot the timing results
fig, ax = plt.subplots(figsize=(10, 6))
ax.semilogy(sizes, times_manual, 'ro-', linewidth=2, markersize=8, label='Manual DFT $O(M^2)$')
ax.semilogy(sizes, times_fft, 'bs-', linewidth=2, markersize=8, label='NumPy FFT $O(M \log_2 M)$')
ax.set_xlabel('Signal Length M', fontsize=12)
ax.set_ylabel('Execution Time (seconds)', fontsize=12)
ax.set_title('Timing Comparison: Manual DFT vs NumPy FFT', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Separability of the 2-D DFT

The 2-D DFT can be decomposed into successive 1-D DFTs thanks to the **separability** property:

$$F(u, v) = \sum_{x=0}^{M-1} \sum_{y=0}^{N-1} f(x, y) \, e^{-j2\pi(ux/M + vy/N)}$$

This can be rewritten as:

$$F(u, v) = \sum_{x=0}^{M-1} e^{-j2\pi ux/M} \left[ \sum_{y=0}^{N-1} f(x, y) \, e^{-j2\pi vy/N} \right]$$

### Two-step computation:
1. **Step 1**: Compute 1-D DFTs along each **row** of $f(x,y)$, producing an intermediate result $F(x, v)$
2. **Step 2**: Compute 1-D DFTs along each **column** of $F(x, v)$, producing the final result $F(u, v)$

This means a 2-D FFT of an $M \times N$ image requires:
- $M$ one-dimensional FFTs of length $N$ (rows)
- $N$ one-dimensional FFTs of length $M$ (columns)

Total: $O(MN \log_2 M + MN \log_2 N) = O(MN \log_2(MN))$ instead of $O(M^2 N^2)$.

In [ ]:
# Demonstrate separability: 2-D DFT as successive 1-D DFTs
M, N = 64, 64
img = np.random.rand(M, N)

# Method 1: Direct 2-D FFT using NumPy
F_direct = np.fft.fft2(img)

# Method 2: Separable computation (row FFTs, then column FFTs)
# Step 1: FFT along each row
F_rows = np.zeros((M, N), dtype=np.complex128)
for x in range(M):
    F_rows[x, :] = np.fft.fft(img[x, :])

# Step 2: FFT along each column of the intermediate result
F_separable = np.zeros((M, N), dtype=np.complex128)
for v in range(N):
    F_separable[:, v] = np.fft.fft(F_rows[:, v])

# Verify they produce the same result
diff = np.max(np.abs(F_direct - F_separable))
print(f'Max difference between fft2 and separable 1-D FFTs: {diff:.2e}')
print('The 2-D DFT is indeed separable into 1-D DFTs!\n')

# Visualize the two-step process
img_demo = np.zeros((64, 64), dtype=np.float64)
img_demo[20:45, 25:40] = 255

F_step1 = np.zeros_like(img_demo, dtype=np.complex128)
for x in range(64):
    F_step1[x, :] = np.fft.fft(img_demo[x, :])

F_step2 = np.zeros_like(img_demo, dtype=np.complex128)
for v in range(64):
    F_step2[:, v] = np.fft.fft(F_step1[:, v])

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(img_demo, cmap='gray')
axes[0].set_title('Original Image\n$f(x,y)$', fontsize=11)
axes[0].axis('off')

axes[1].imshow(np.log1p(np.abs(F_step1)), cmap='gray')
axes[1].set_title('Step 1: Row FFTs\n$F(x, v)$', fontsize=11)
axes[1].axis('off')

axes[2].imshow(np.log1p(np.abs(np.fft.fftshift(F_step2))), cmap='gray')
axes[2].set_title('Step 2: Column FFTs\n$F(u, v)$ (centered)', fontsize=11)
axes[2].axis('off')

axes[3].imshow(np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(img_demo)))), cmap='gray')
axes[3].set_title('Direct fft2 (centered)\n(Same result)', fontsize=11)
axes[3].axis('off')

plt.suptitle('Separability: 2-D DFT = Row FFTs + Column FFTs', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Unsharp Masking in the Frequency Domain

**Unsharp masking** is a classical sharpening technique. The idea is simple:

1. Blur the image (lowpass filter) to get a smooth version $f_{LP}(x,y)$
2. Subtract the smooth version from the original to extract the **high-frequency detail** (the "mask")
3. Add a scaled version of the detail back to the original

### Spatial domain formulation:

$$g(x,y) = f(x,y) + k \cdot \underbrace{\left[ f(x,y) - f_{LP}(x,y) \right]}_{\text{highpass detail (unsharp mask)}}$$

Where $k > 0$ controls the sharpening strength.

### Frequency domain formulation:

Since $f_{LP}(x,y)$ is obtained by applying a lowpass filter $H_{LP}(u,v)$:

$$F_{LP}(u,v) = H_{LP}(u,v) \cdot F(u,v)$$

The output in the frequency domain becomes:

$$G(u,v) = F(u,v) + k \cdot \left[ F(u,v) - H_{LP}(u,v) \cdot F(u,v) \right]$$

$$\boxed{G(u,v) = \underbrace{\left[ 1 + k - k \cdot H_{LP}(u,v) \right]}_{H_{\text{unsharp}}(u,v)} \cdot F(u,v)}$$

The effective filter is:

$$H_{\text{unsharp}}(u,v) = 1 + k - k \cdot H_{LP}(u,v)$$

- When $k = 1$: standard unsharp masking
- When $k > 1$: high-boost filtering (stronger sharpening)

In [ ]:
# Create a synthetic biomedical-style test image
M, N = 256, 256
Y, X = np.mgrid[-M//2:M//2, -N//2:N//2]

# Simulate cell-like structures
np.random.seed(42)
img = np.zeros((M, N), dtype=np.float64)

# Background with gentle gradient
img += 80

# Add several circular "cells"
cell_centers = [(30, 40), (-50, 60), (20, -70), (-30, -40), (60, 0),
                (-70, 50), (0, 0), (50, -50), (-20, 80), (70, -70)]
cell_radii = [20, 25, 18, 22, 15, 20, 30, 16, 19, 21]
cell_brightness = [180, 200, 170, 190, 160, 175, 210, 165, 185, 195]

for (cy, cx), r, b in zip(cell_centers, cell_radii, cell_brightness):
    mask = (Y - cy)**2 + (X - cx)**2 <= r**2
    img[mask] = b
    # Add brighter nucleus
    nucleus_mask = (Y - cy)**2 + (X - cx)**2 <= (r * 0.4)**2
    img[nucleus_mask] = min(b + 40, 255)

# Add slight blur to make it more realistic
from numpy.fft import fft2, ifft2, fftshift, ifftshift, fftfreq

# Slight Gaussian blur
D0_blur = 80
D = np.sqrt(X**2 + Y**2)
H_blur = np.exp(-(D**2) / (2 * D0_blur**2))
F_img = fftshift(fft2(img))
img = np.real(ifft2(ifftshift(F_img * H_blur)))
img = np.clip(img, 0, 255)

plt.figure(figsize=(6, 6))
plt.imshow(img, cmap='gray', vmin=0, vmax=255)
plt.title('Test Image: Synthetic Cell Structures', fontsize=13)
plt.colorbar()
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
def gaussian_lowpass(shape, D0):
    """Create a Gaussian lowpass filter H(u,v) = exp(-D^2 / (2*D0^2))."""
    M, N = shape
    u = np.arange(M) - M // 2
    v = np.arange(N) - N // 2
    V, U = np.meshgrid(v, u)
    D = np.sqrt(U**2 + V**2)
    return np.exp(-(D**2) / (2 * D0**2))

def unsharp_mask_freq(image, D0=30, k=1.0):
    """
    Apply unsharp masking in the frequency domain.
    
    H_unsharp(u,v) = 1 + k - k * H_LP(u,v)
    
    Parameters:
    - image: input image (2D array)
    - D0: cutoff frequency for the Gaussian lowpass filter
    - k: sharpening strength (k=1: unsharp mask, k>1: high-boost)
    
    Returns:
    - sharpened image, the unsharp filter H, lowpass filter H_LP
    """
    M, N = image.shape
    
    # Compute centered DFT
    F = fftshift(fft2(image))
    
    # Create Gaussian lowpass filter
    H_LP = gaussian_lowpass((M, N), D0)
    
    # Construct unsharp masking filter
    H_unsharp = 1.0 + k - k * H_LP
    
    # Apply filter
    G = H_unsharp * F
    
    # Inverse DFT
    result = np.real(ifft2(ifftshift(G)))
    result = np.clip(result, 0, 255)
    
    return result, H_unsharp, H_LP

# Apply with different parameters
result_k1, H_unsharp_k1, H_LP = unsharp_mask_freq(img, D0=30, k=1.0)
result_k2, H_unsharp_k2, _ = unsharp_mask_freq(img, D0=30, k=2.0)
result_k3, H_unsharp_k3, _ = unsharp_mask_freq(img, D0=30, k=3.0)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Top row: images
axes[0, 0].imshow(img, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original Image', fontsize=11)
axes[0, 0].axis('off')

axes[0, 1].imshow(result_k1, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title('Unsharp Mask (k=1)', fontsize=11)
axes[0, 1].axis('off')

axes[0, 2].imshow(result_k2, cmap='gray', vmin=0, vmax=255)
axes[0, 2].set_title('High-Boost (k=2)', fontsize=11)
axes[0, 2].axis('off')

axes[0, 3].imshow(result_k3, cmap='gray', vmin=0, vmax=255)
axes[0, 3].set_title('High-Boost (k=3)', fontsize=11)
axes[0, 3].axis('off')

# Bottom row: filter cross-sections
center = img.shape[0] // 2
freqs = np.arange(img.shape[1]) - img.shape[1] // 2

axes[1, 0].plot(freqs, H_LP[center, :], 'b-', linewidth=2)
axes[1, 0].set_title('$H_{LP}$ (Gaussian Lowpass)', fontsize=11)
axes[1, 0].set_xlabel('Frequency')
axes[1, 0].set_ylabel('$H(u,v)$')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 3.5])

axes[1, 1].plot(freqs, H_unsharp_k1[center, :], 'r-', linewidth=2)
axes[1, 1].set_title('$H_{unsharp}$ (k=1)', fontsize=11)
axes[1, 1].set_xlabel('Frequency')
axes[1, 1].grid(True, alpha=0.3)
axes[1, 1].set_ylim([0, 3.5])

axes[1, 2].plot(freqs, H_unsharp_k2[center, :], 'r-', linewidth=2)
axes[1, 2].set_title('$H_{unsharp}$ (k=2)', fontsize=11)
axes[1, 2].set_xlabel('Frequency')
axes[1, 2].grid(True, alpha=0.3)
axes[1, 2].set_ylim([0, 3.5])

axes[1, 3].plot(freqs, H_unsharp_k3[center, :], 'r-', linewidth=2)
axes[1, 3].set_title('$H_{unsharp}$ (k=3)', fontsize=11)
axes[1, 3].set_xlabel('Frequency')
axes[1, 3].grid(True, alpha=0.3)
axes[1, 3].set_ylim([0, 4.5])

plt.suptitle('Unsharp Masking in the Frequency Domain\n$G = [1 + k - k \cdot H_{LP}] \cdot F$',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Show the decomposition: original = lowpass + highpass detail
F = fftshift(fft2(img))
H_LP = gaussian_lowpass(img.shape, D0=30)

# Lowpass component
img_lp = np.real(ifft2(ifftshift(H_LP * F)))

# Highpass detail (unsharp mask)
detail = img - img_lp

# Sharpened = original + k * detail
k = 1.5
sharpened = np.clip(img + k * detail, 0, 255)

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

axes[0].imshow(img, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Original $f(x,y)$', fontsize=12)
axes[0].axis('off')

axes[1].imshow(img_lp, cmap='gray', vmin=0, vmax=255)
axes[1].set_title('Lowpass $f_{LP}(x,y)$', fontsize=12)
axes[1].axis('off')

axes[2].imshow(detail, cmap='gray')
axes[2].set_title('Detail: $f - f_{LP}$\n(the "unsharp mask")', fontsize=12)
axes[2].axis('off')

axes[3].imshow(sharpened, cmap='gray', vmin=0, vmax=255)
axes[3].set_title(f'Sharpened: $f + {k} \\cdot (f - f_{{LP}})$', fontsize=12)
axes[3].axis('off')

plt.suptitle('Unsharp Masking Decomposition', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Homomorphic Filtering

### The Illumination-Reflectance Model

An image can be modeled as the product of two components:

$$f(x,y) = i(x,y) \cdot r(x,y)$$

Where:
- $i(x,y)$ = **illumination** component (light falling on the scene)
- $r(x,y)$ = **reflectance** component (how much light is reflected by objects)

### Key observations:
- **Illumination** varies **slowly** (low frequencies) -- it comes from the light source
- **Reflectance** varies **rapidly** (high frequencies) -- it encodes object edges and textures
- Typical ranges: $0 < i(x,y) < \infty$, $\quad 0 < r(x,y) < 1$

### The Problem

We cannot directly apply the Fourier transform to separate these components because they are **multiplied**, not added. The DFT is a **linear** operation -- it distributes over addition, not multiplication.

### The Solution: Logarithm

Take the natural logarithm to convert multiplication into addition:

$$\ln f(x,y) = \ln i(x,y) + \ln r(x,y)$$

Now we define: $z(x,y) = \ln f(x,y)$, and in the frequency domain:

$$Z(u,v) = F_i(u,v) + F_r(u,v)$$

Where $F_i = \mathcal{F}\{\ln i\}$ and $F_r = \mathcal{F}\{\ln r\}$.

### Homomorphic Filter Pipeline

$$f(x,y) \xrightarrow{\ln} z(x,y) \xrightarrow{\text{DFT}} Z(u,v) \xrightarrow{H(u,v)} S(u,v) \xrightarrow{\text{IDFT}} s(x,y) \xrightarrow{\exp} g(x,y)$$

### The Homomorphic Filter

We use a filter that:
- **Compresses** the illumination range (attenuate low frequencies with $\gamma_L < 1$)
- **Enhances** the reflectance/detail (amplify high frequencies with $\gamma_H > 1$)

$$H(u,v) = (\gamma_H - \gamma_L) \left[ 1 - e^{-c \left( D^2(u,v) / D_0^2 \right)} \right] + \gamma_L$$

Where:
- $D(u,v)$ = distance from center in frequency domain
- $D_0$ = cutoff frequency
- $c$ = controls filter sharpness
- $\gamma_L$ = gain for low frequencies (typically $< 1$)
- $\gamma_H$ = gain for high frequencies (typically $> 1$)

At $D = 0$: $H = \gamma_L$ (low frequencies attenuated).
At $D \to \infty$: $H \to \gamma_H$ (high frequencies amplified).

In [ ]:
# Visualize the homomorphic filter shape
def homomorphic_filter(shape, D0, gamma_L, gamma_H, c=1.0):
    """
    Create a homomorphic filter.
    
    H(u,v) = (gamma_H - gamma_L) * [1 - exp(-c * D^2 / D0^2)] + gamma_L
    
    Parameters:
    - shape: (M, N) image dimensions
    - D0: cutoff frequency
    - gamma_L: low frequency gain (< 1 to compress illumination)
    - gamma_H: high frequency gain (> 1 to enhance reflectance)
    - c: controls filter transition sharpness
    
    Returns:
    - H: the homomorphic filter (M x N array)
    """
    M, N = shape
    u = np.arange(M) - M // 2
    v = np.arange(N) - N // 2
    V, U = np.meshgrid(v, u)
    D = np.sqrt(U**2 + V**2)
    
    H = (gamma_H - gamma_L) * (1 - np.exp(-c * (D**2 / D0**2))) + gamma_L
    return H

# Show filter for different parameter settings
D0 = 30
D_1d = np.linspace(0, 128, 500)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Different gamma_L and gamma_H
params = [
    (0.5, 2.0, 1.0, 'Standard'),
    (0.3, 2.5, 1.0, 'Stronger'),
    (0.5, 2.0, 0.5, 'Smoother (c=0.5)')
]

for ax, (gL, gH, c, label) in zip(axes, params):
    H_1d = (gH - gL) * (1 - np.exp(-c * (D_1d**2 / D0**2))) + gL
    ax.plot(D_1d, H_1d, 'b-', linewidth=2)
    ax.axhline(y=gL, color='r', linestyle='--', alpha=0.7, label=f'$\\gamma_L = {gL}$')
    ax.axhline(y=gH, color='g', linestyle='--', alpha=0.7, label=f'$\\gamma_H = {gH}$')
    ax.axhline(y=1.0, color='k', linestyle=':', alpha=0.5, label='Unity gain')
    ax.axvline(x=D0, color='orange', linestyle='--', alpha=0.7, label=f'$D_0 = {D0}$')
    ax.set_xlabel('$D(u,v)$', fontsize=11)
    ax.set_ylabel('$H(u,v)$', fontsize=11)
    ax.set_title(f'{label}\n$\\gamma_L={gL}, \\gamma_H={gH}, c={c}$', fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 3.0])

plt.suptitle('Homomorphic Filter: $H = (\\gamma_H - \\gamma_L)[1 - e^{-c(D^2/D_0^2)}] + \\gamma_L$',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Create a test image with UNEVEN ILLUMINATION
M, N = 256, 256
Y, X = np.mgrid[-M//2:M//2, -N//2:N//2]

# Reflectance component: cell-like structures with sharp edges
reflectance = np.ones((M, N)) * 0.5  # base reflectance

# Add circular "cells" with different reflectance
cell_params = [
    (40, 50, 20, 0.8), (-60, 70, 22, 0.85), (30, -60, 18, 0.75),
    (-40, -50, 25, 0.9), (70, -10, 16, 0.7), (-20, 0, 28, 0.82),
    (0, -90, 19, 0.78), (-80, -20, 21, 0.88), (50, 80, 17, 0.72),
    (-10, -70, 23, 0.83)
]

for cy, cx, r, ref in cell_params:
    mask = (Y - cy)**2 + (X - cx)**2 <= r**2
    reflectance[mask] = ref
    # Nucleus
    nucleus_mask = (Y - cy)**2 + (X - cx)**2 <= (r * 0.35)**2
    reflectance[nucleus_mask] = min(ref + 0.1, 1.0)

# Illumination component: strong uneven lighting (bright top-left, dark bottom-right)
illumination = 200 * np.exp(-((Y - (-80))**2 + (X - (-80))**2) / (2 * 120**2))
illumination += 50  # minimum illumination

# Form the image: f = illumination * reflectance
img_uneven = illumination * reflectance
img_uneven = np.clip(img_uneven, 1, 255)  # avoid log(0)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(illumination, cmap='gray')
axes[0].set_title('Illumination $i(x,y)$\n(varies slowly)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(reflectance, cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Reflectance $r(x,y)$\n(varies rapidly)', fontsize=12)
axes[1].axis('off')

axes[2].imshow(img_uneven, cmap='gray', vmin=0, vmax=255)
axes[2].set_title('Image $f = i \\cdot r$\n(uneven illumination)', fontsize=12)
axes[2].axis('off')

plt.suptitle('Image Formation: $f(x,y) = i(x,y) \\cdot r(x,y)$', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def apply_homomorphic_filter(image, D0=30, gamma_L=0.5, gamma_H=2.0, c=1.0):
    """
    Apply homomorphic filtering to correct uneven illumination.
    
    Pipeline: f -> ln -> DFT -> H(u,v) -> IDFT -> exp -> g
    
    Parameters:
    - image: input image
    - D0: cutoff frequency
    - gamma_L: low frequency gain (< 1)
    - gamma_H: high frequency gain (> 1)
    - c: filter sharpness
    
    Returns:
    - filtered image, intermediate results for visualization
    """
    # Step 1: Take natural log (add small epsilon to avoid log(0))
    img_log = np.log(image.astype(np.float64) + 1.0)
    
    # Step 2: Compute DFT
    F = fftshift(fft2(img_log))
    
    # Step 3: Create and apply homomorphic filter
    H = homomorphic_filter(image.shape, D0, gamma_L, gamma_H, c)
    G = H * F
    
    # Step 4: Inverse DFT
    g_log = np.real(ifft2(ifftshift(G)))
    
    # Step 5: Take exponential to reverse the log
    g = np.exp(g_log) - 1.0
    
    # Normalize to [0, 255]
    g = (g - g.min()) / (g.max() - g.min()) * 255.0
    
    return g, img_log, H

# Apply homomorphic filtering
result_homo, img_log, H_homo = apply_homomorphic_filter(
    img_uneven, D0=30, gamma_L=0.4, gamma_H=2.0, c=1.0
)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Top row: pipeline
axes[0, 0].imshow(img_uneven, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('1. Original $f(x,y)$\n(uneven illumination)', fontsize=11)
axes[0, 0].axis('off')

axes[0, 1].imshow(img_log, cmap='gray')
axes[0, 1].set_title('2. Log transform\n$z = \\ln(f)$', fontsize=11)
axes[0, 1].axis('off')

axes[0, 2].imshow(np.log1p(np.abs(fftshift(fft2(img_log)))), cmap='gray')
axes[0, 2].set_title('3. Spectrum of $\\ln(f)$', fontsize=11)
axes[0, 2].axis('off')

# Bottom row
axes[1, 0].imshow(H_homo, cmap='jet')
axes[1, 0].set_title('4. Homomorphic Filter $H(u,v)$\n$\\gamma_L=0.4, \\gamma_H=2.0$', fontsize=11)
axes[1, 0].axis('off')
plt.colorbar(axes[1, 0].images[0], ax=axes[1, 0], fraction=0.046)

axes[1, 1].imshow(result_homo, cmap='gray', vmin=0, vmax=255)
axes[1, 1].set_title('5. Result $g(x,y)$\n(illumination corrected)', fontsize=11)
axes[1, 1].axis('off')

# Compare original vs result side by side
comparison = np.hstack([img_uneven / img_uneven.max() * 255,
                        result_homo / result_homo.max() * 255])
axes[1, 2].imshow(comparison, cmap='gray', vmin=0, vmax=255)
axes[1, 2].set_title('6. Before (left) vs After (right)', fontsize=11)
axes[1, 2].axvline(x=img_uneven.shape[1], color='r', linewidth=2)
axes[1, 2].axis('off')

plt.suptitle('Homomorphic Filtering Pipeline\n$f \\to \\ln \\to DFT \\to H(u,v) \\to IDFT \\to \\exp \\to g$',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Compare different homomorphic filter parameters
params_list = [
    {'D0': 30, 'gamma_L': 0.5, 'gamma_H': 1.5, 'c': 1.0},
    {'D0': 30, 'gamma_L': 0.4, 'gamma_H': 2.0, 'c': 1.0},
    {'D0': 30, 'gamma_L': 0.3, 'gamma_H': 2.5, 'c': 1.0},
    {'D0': 20, 'gamma_L': 0.4, 'gamma_H': 2.0, 'c': 1.0},
    {'D0': 50, 'gamma_L': 0.4, 'gamma_H': 2.0, 'c': 1.0},
    {'D0': 30, 'gamma_L': 0.4, 'gamma_H': 2.0, 'c': 0.3},
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Original
axes[0, 0].imshow(img_uneven, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Original\n(uneven illumination)', fontsize=10)
axes[0, 0].axis('off')

axes[1, 0].imshow(img_uneven, cmap='gray', vmin=0, vmax=255)
axes[1, 0].set_title('Original\n(uneven illumination)', fontsize=10)
axes[1, 0].axis('off')

for idx, params in enumerate(params_list):
    result, _, _ = apply_homomorphic_filter(img_uneven, **params)
    row = idx // 3
    col = (idx % 3) + 1
    axes[row, col].imshow(result, cmap='gray', vmin=0, vmax=255)
    title = (f"$D_0$={params['D0']}, $\\gamma_L$={params['gamma_L']}\n"
             f"$\\gamma_H$={params['gamma_H']}, c={params['c']}")
    axes[row, col].set_title(title, fontsize=10)
    axes[row, col].axis('off')

plt.suptitle('Homomorphic Filtering: Effect of Different Parameters', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Interpretation of Homomorphic Filter Parameters

| Parameter | Effect |
|-----------|--------|
| $\gamma_L < 1$ | Compresses the dynamic range of illumination (makes dark areas brighter) |
| $\gamma_H > 1$ | Enhances reflectance/detail (sharpens edges) |
| $D_0$ | Controls which frequencies are considered "low" vs "high" |
| $c$ | Controls the steepness of the transition between $\gamma_L$ and $\gamma_H$ |

**Smaller $\gamma_L$** = more illumination compression (more uniform brightness).

**Larger $\gamma_H$** = more detail enhancement (but may amplify noise).

**Smaller $D_0$** = broader "low frequency" band (more aggressive illumination correction).

## 5. Application: Complete Image Enhancement Pipeline

In practice, frequency domain filtering follows a systematic pipeline:

1. **Load/Create** the input image
2. **Analyze** the frequency spectrum to understand the image content
3. **Choose** an appropriate filter based on the analysis
4. **Apply** the filter in the frequency domain
5. **Compare** the results

Let us demonstrate a complete pipeline that combines multiple techniques.

In [ ]:
# ============================================================
# STEP 1: Create a challenging test image
# ============================================================
# Simulate a biomedical microscopy image with:
#   - Uneven illumination
#   - Fine detail (cell structures)
#   - Noise

M, N = 256, 256
Y, X = np.mgrid[-M//2:M//2, -N//2:N//2]

np.random.seed(123)

# Create reflectance: tissue-like structures
reflectance = np.ones((M, N)) * 0.4

# Larger "tissue" regions
for _ in range(5):
    cy, cx = np.random.randint(-80, 80, 2)
    ry, rx = np.random.randint(30, 60, 2)
    mask = ((Y - cy) / ry)**2 + ((X - cx) / rx)**2 <= 1
    reflectance[mask] = np.random.uniform(0.55, 0.75)

# Smaller "cells" within tissue
for _ in range(25):
    cy, cx = np.random.randint(-100, 100, 2)
    r = np.random.randint(5, 14)
    mask = (Y - cy)**2 + (X - cx)**2 <= r**2
    reflectance[mask] = np.random.uniform(0.7, 0.95)
    # Nucleus
    nucleus_mask = (Y - cy)**2 + (X - cx)**2 <= (r * 0.3)**2
    reflectance[nucleus_mask] = np.random.uniform(0.85, 1.0)

# Illumination: strongly uneven (simulating microscope light)
illumination = 180 * np.exp(-((Y + 40)**2 + (X - 30)**2) / (2 * 100**2))
illumination += 40

# Form the image
img_raw = illumination * reflectance

# Add Gaussian noise
noise = np.random.randn(M, N) * 8
img_noisy = np.clip(img_raw + noise, 1, 255)

print('Step 1: Test image created.')
print(f'  Size: {M} x {N}')
print(f'  Intensity range: [{img_noisy.min():.1f}, {img_noisy.max():.1f}]')
print(f'  Mean intensity: {img_noisy.mean():.1f}')

plt.figure(figsize=(7, 7))
plt.imshow(img_noisy, cmap='gray', vmin=0, vmax=255)
plt.title('Input Image: Noisy with Uneven Illumination', fontsize=13)
plt.colorbar()
plt.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 2: Analyze the frequency spectrum
# ============================================================
F_input = fftshift(fft2(img_noisy))
spectrum_input = np.log1p(np.abs(F_input))

# Compute radial average of the power spectrum
D = np.sqrt(X**2 + Y**2)
power = np.abs(F_input)**2

# Radial average
max_radius = int(np.sqrt((M//2)**2 + (N//2)**2))
radial_profile = np.zeros(max_radius)
radial_count = np.zeros(max_radius)
for r in range(max_radius):
    ring = (D >= r) & (D < r + 1)
    if np.any(ring):
        radial_profile[r] = np.mean(power[ring])
        radial_count[r] = np.sum(ring)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(img_noisy, cmap='gray', vmin=0, vmax=255)
axes[0].set_title('Input Image', fontsize=12)
axes[0].axis('off')

axes[1].imshow(spectrum_input, cmap='gray')
axes[1].set_title('Centered Spectrum\n$\\log(1 + |F(u,v)|)$', fontsize=12)
axes[1].axis('off')

axes[2].semilogy(radial_profile[:max_radius//2], 'b-', linewidth=1.5)
axes[2].set_title('Radial Power Spectrum', fontsize=12)
axes[2].set_xlabel('Frequency (distance from center)', fontsize=11)
axes[2].set_ylabel('Average Power', fontsize=11)
axes[2].grid(True, alpha=0.3)
axes[2].axvspan(0, 15, alpha=0.2, color='red', label='Low freq (illumination)')
axes[2].axvspan(80, max_radius//2, alpha=0.2, color='green', label='High freq (noise)')
axes[2].legend(fontsize=9)

plt.suptitle('Step 2: Spectrum Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Spectrum analysis:')
print('  - Strong DC component (bright center) -> high average intensity')
print('  - Low frequency energy concentration -> uneven illumination')
print('  - Flat high frequency floor -> noise')

In [ ]:
# ============================================================
# STEP 3 & 4: Choose and apply filters
# ============================================================

# Strategy:
# 1. First, apply a gentle lowpass filter to reduce noise
# 2. Then, apply homomorphic filtering to correct illumination
# 3. Finally, apply unsharp masking to enhance detail

# --- Stage A: Noise reduction with Gaussian lowpass ---
D0_denoise = 60  # keep most detail, just cut high-freq noise
H_denoise = gaussian_lowpass(img_noisy.shape, D0_denoise)
F_noisy = fftshift(fft2(img_noisy))
img_denoised = np.real(ifft2(ifftshift(H_denoise * F_noisy)))
img_denoised = np.clip(img_denoised, 1, 255)

# --- Stage B: Illumination correction with homomorphic filter ---
img_homo, _, H_homo_pipeline = apply_homomorphic_filter(
    img_denoised, D0=30, gamma_L=0.4, gamma_H=1.8, c=1.0
)

# --- Stage C: Detail enhancement with unsharp masking ---
img_sharp, _, _ = unsharp_mask_freq(img_homo, D0=40, k=1.5)

print('Filters applied:')
print(f'  Stage A: Gaussian lowpass (D0={D0_denoise}) for noise reduction')
print(f'  Stage B: Homomorphic filter (gamma_L=0.4, gamma_H=1.8) for illumination correction')
print(f'  Stage C: Unsharp masking (k=1.5) for detail enhancement')

In [ ]:
# ============================================================
# STEP 5: Compare results at each stage
# ============================================================

fig, axes = plt.subplots(2, 4, figsize=(18, 9))

# Top row: images at each stage
images_pipeline = [img_noisy, img_denoised, img_homo, img_sharp]
titles_pipeline = [
    'Input Image\n(noisy, uneven illumination)',
    'Stage A: Denoised\n(Gaussian lowpass)',
    'Stage B: Illumination Corrected\n(Homomorphic filter)',
    'Stage C: Detail Enhanced\n(Unsharp masking)'
]

for i, (im, title) in enumerate(zip(images_pipeline, titles_pipeline)):
    axes[0, i].imshow(im, cmap='gray', vmin=0, vmax=255)
    axes[0, i].set_title(title, fontsize=10)
    axes[0, i].axis('off')

# Bottom row: spectra at each stage
for i, im in enumerate(images_pipeline):
    spec = np.log1p(np.abs(fftshift(fft2(im))))
    axes[1, i].imshow(spec, cmap='gray')
    axes[1, i].set_title('Spectrum', fontsize=10)
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('Image', fontsize=12)
axes[1, 0].set_ylabel('Spectrum', fontsize=12)

plt.suptitle('Complete Enhancement Pipeline: Input -> Denoise -> Illumination -> Sharpen',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Final comparison: input vs output with intensity profiles
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(img_noisy, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('Input Image', fontsize=13)
axes[0, 0].axhline(y=M//2, color='r', linestyle='--', linewidth=1.5, alpha=0.7)
axes[0, 0].axis('off')

axes[0, 1].imshow(img_sharp, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title('Enhanced Output', fontsize=13)
axes[0, 1].axhline(y=M//2, color='r', linestyle='--', linewidth=1.5, alpha=0.7)
axes[0, 1].axis('off')

# Horizontal intensity profile through the middle row
row = M // 2
axes[1, 0].plot(img_noisy[row, :], 'b-', alpha=0.7, label='Input')
axes[1, 0].plot(img_sharp[row, :], 'r-', linewidth=2, label='Enhanced')
axes[1, 0].set_title(f'Intensity Profile (row {row})', fontsize=13)
axes[1, 0].set_xlabel('Column', fontsize=11)
axes[1, 0].set_ylabel('Intensity', fontsize=11)
axes[1, 0].legend(fontsize=11)
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_ylim([0, 280])

# Histograms
axes[1, 1].hist(img_noisy.ravel(), bins=64, alpha=0.5, color='blue',
                label='Input', density=True)
axes[1, 1].hist(img_sharp.ravel(), bins=64, alpha=0.5, color='red',
                label='Enhanced', density=True)
axes[1, 1].set_title('Intensity Histograms', fontsize=13)
axes[1, 1].set_xlabel('Intensity', fontsize=11)
axes[1, 1].set_ylabel('Normalized Frequency', fontsize=11)
axes[1, 1].legend(fontsize=11)
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Final Comparison: Input vs Enhanced Output',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('Enhancement results:')
print(f'  Input  - mean: {img_noisy.mean():.1f}, std: {img_noisy.std():.1f}')
print(f'  Output - mean: {img_sharp.mean():.1f}, std: {img_sharp.std():.1f}')
print()
print('The enhanced image shows:')
print('  - Reduced noise (smoother background)')
print('  - More uniform illumination (brighter in previously dark areas)')
print('  - Enhanced detail (sharper cell boundaries)')
print('  - Better contrast distribution (wider histogram)')

## Summary

What we learned:

1. **Fast Fourier Transform (FFT)**: Reduces DFT complexity from $O(M^2)$ to $O(M \log_2 M)$ by exploiting symmetry and periodicity of the complex exponential. This makes frequency domain processing practical for real-world images.

2. **Separability**: The 2-D DFT can be computed as 1-D FFTs along rows followed by 1-D FFTs along columns, further leveraging the efficiency of the FFT.

3. **Unsharp Masking**: Sharpening by subtracting a lowpass version and adding back scaled detail. In the frequency domain: $G = [1 + k - k \cdot H_{LP}] \cdot F$. The parameter $k$ controls sharpening strength.

4. **Homomorphic Filtering**: Separates illumination (low freq) from reflectance (high freq) using the log transform. The filter $H = (\gamma_H - \gamma_L)[1 - e^{-c(D^2/D_0^2)}] + \gamma_L$ compresses illumination range while enhancing detail.

5. **Enhancement Pipeline**: Real-world image enhancement often combines multiple filters in sequence: noise reduction (lowpass) -> illumination correction (homomorphic) -> detail enhancement (unsharp masking).